In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv("popular_anime.csv")
print("Raw shape :", df.shape)

In [ ]:
# ---------- Prepare the analysis subset ----------
data = df.dropna(subset=["score", "scored_by", "episodes", "type", "rank"]).copy()
data["popularity"]  = np.log10(data["scored_by"])
data["decade"]      = (pd.to_datetime(data["aired_from"], errors="coerce", utc=True)
                         .dt.year // 10 * 10)
data = data.dropna(subset=["decade"])
data["decade"] = data["decade"].astype(int)

print("Analysis shape :", data.shape)
print(data[["name", "type", "score", "popularity", "decade"]].head())

In [ ]:
# ---------- Grouped aggregation ----------
by_type = data.groupby("type")["score"].agg(["count", "mean", "median", "std"]).round(3)
print(by_type.sort_values("mean", ascending=False))

In [ ]:
# ---------- Multiple aggregations at once ----------
multi = data.groupby("type").agg(
    titles     = ("name", "count"),
    avg_score  = ("score", "mean"),
    max_score  = ("score", "max"),
    avg_eps    = ("episodes", "mean"),
    total_votes= ("scored_by", "sum")
).round(2)

print(multi.sort_values("titles", ascending=False))

In [ ]:
# ---------- Pivot table ----------
pivot = pd.pivot_table(data[data["decade"] >= 1980],
                       values="score", index="decade", columns="type",
                       aggfunc="mean").round(2)

print(pivot[["TV", "Movie", "OVA", "Special"]])

In [ ]:
# ---------- Cross tabulation ----------
data["band"] = pd.cut(data["score"], [0, 6, 7, 8, 10],
                      labels=["Low", "Medium", "High", "Very High"])
ct = pd.crosstab(data["type"], data["band"])

print(ct)

In [ ]:
# ---------- Retain only the significant groups ----------
studio = (data.dropna(subset=["studios"])
              .groupby("studios")["score"]
              .agg(["count", "mean"])
              .query("count >= 50")
              .sort_values("mean", ascending=False)
              .round(3))

print("Studios with at least 50 titles :", len(studio))
print(studio.head(8))

In [ ]:
# ---------- Pearson correlation with significance ----------
r, p = stats.pearsonr(data["popularity"], data["score"])

print("Pearson r :", round(r, 4))
print("p value   :", "%.3e" % p)
print("Significant at 5 percent :", p < 0.05)

In [ ]:
# ---------- Spearman and Kendall correlation ----------
rs, ps = stats.spearmanr(data["popularity"], data["score"])
rk, pk = stats.kendalltau(data["popularity"], data["score"])

print("Spearman rho :", round(rs, 4), "  p value :", "%.3e" % ps)
print("Kendall  tau :", round(rk, 4), "  p value :", "%.3e" % pk)

In [ ]:
# ---------- Compare the three coefficients ----------
comp = pd.DataFrame({
    "Method"      : ["Pearson", "Spearman", "Kendall"],
    "Coefficient" : [round(r, 4), round(rs, 4), round(rk, 4)],
    "Measures"    : ["Linear relationship",
                     "Monotonic relationship",
                     "Concordance of ranks"]
})
print(comp.to_string(index=False))

In [ ]:
# ---------- Full correlation matrix of derived features ----------
feat = data[["score", "popularity", "episodes", "rank"]]
corr = feat.corr().round(4)

print(corr)

In [ ]:
# ---------- Significance of every pair ----------
cols = feat.columns
print("Pair".ljust(28), "r".rjust(9), "p value".rjust(13))
print("-" * 52)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        rr, pp = stats.pearsonr(feat[cols[i]], feat[cols[j]])
        pair = cols[i] + " vs " + cols[j]
        print(pair.ljust(28), str(round(rr, 4)).rjust(9), ("%.2e" % pp).rjust(13))

In [ ]:
# ---------- Scatter plot with the fitted regression line ----------
s = data.sample(3000, random_state=1)
m, c = np.polyfit(s["popularity"], s["score"], 1)
xs = np.linspace(s["popularity"].min(), s["popularity"].max(), 100)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(s["popularity"], s["score"], s=8, alpha=0.25, color="#3b7dd8")
ax.plot(xs, m * xs + c, color="#d1495b", linewidth=2,
        label="y = %.3fx + %.3f" % (m, c))

ax.set_xlabel("Popularity  (log10 of number of votes)")
ax.set_ylabel("Score")
ax.set_title("Score against Popularity  (r = %.3f)" % r)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()